# 03 - Data quality

**Audience:** decide whether a recording is worth analyzing.

Quality is the gate. Bad data has to be caught here, before fixation
detection or LHIPA gives you false confidence in noise.

## What you'll get

- Validity %, sample-rate stability, gap distribution.
- Per-eye coverage.
- Pupil-range diagnostic (rules out sensor errors).
- Blink rate (sanity-check against literature: 10-30/min relaxed,
  fewer under cognitive load).
- A composite quality score for thresholded batch filtering.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Compute

In [ ]:
df = ctx.data.df
ts = ctx.data.get_timestamps()
qm = ela.calculate_quality_metrics(
    timestamps=ts,
    validity=df["is_tracking_valid"].to_numpy() if "is_tracking_valid" in df.columns else None,
    left_validity=df["has_left_direction"].to_numpy() if "has_left_direction" in df.columns else None,
    right_validity=df["has_right_direction"].to_numpy() if "has_right_direction" in df.columns else None,
    left_pupil=df["left_pupil_diameter"].to_numpy() if "left_pupil_diameter" in df.columns else None,
    right_pupil=df["right_pupil_diameter"].to_numpy() if "right_pupil_diameter" in df.columns else None,
    left_openness=df["left_openness"].to_numpy() if "left_openness" in df.columns else None,
    right_openness=df["right_openness"].to_numpy() if "right_openness" in df.columns else None,
)

## 2. Top-line summary

In [ ]:
for k, v in qm.to_dict().items():
    if isinstance(v, float):
        print(f"  {k:<32} {v:.3f}")
    else:
        print(f"  {k:<32} {v}")

## 3. Sample-rate stability

In [ ]:
deltas = np.diff(ts) * 1000  # ms
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(deltas, bins=80, range=(0, np.percentile(deltas, 99) * 1.2))
ax.set_xlabel("sample interval (ms)")
ax.set_ylabel("count")
ax.set_title(f"Sample-interval distribution  (mean {np.mean(deltas):.2f} ms)")
plt.tight_layout()
plt.show()

## 4. Validity over time

In [ ]:
if "is_tracking_valid" in df.columns:
    valid = df["is_tracking_valid"].astype(bool).to_numpy()
    # 1-second moving validity
    sr = ctx.data.get_sample_rate() or 90.0
    win = int(round(sr))
    if win > 1 and len(valid) > win:
        kernel = np.ones(win) / win
        rolling = np.convolve(valid.astype(float), kernel, mode="same")
        fig, ax = plt.subplots(figsize=(12, 2.5))
        ax.plot(ts, rolling, linewidth=0.7)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("time (s)")
        ax.set_ylabel("1 s rolling validity")
        ax.set_title("Tracking validity over time")
        plt.tight_layout()
        plt.show()
else:
    print("No is_tracking_valid column.")

## 5. Pupil-range sanity check

In [ ]:
have_left = "left_pupil_diameter" in df.columns
have_right = "right_pupil_diameter" in df.columns
if have_left or have_right:
    fig, ax = plt.subplots(figsize=(10, 3))
    if have_left:
        l = df["left_pupil_diameter"].dropna()
        ax.hist(l, bins=60, alpha=0.6, label=f"left ({l.median():.2f} mm med)")
    if have_right:
        r = df["right_pupil_diameter"].dropna()
        ax.hist(r, bins=60, alpha=0.6, label=f"right ({r.median():.2f} mm med)")
    ax.set_xlabel("pupil diameter (mm)")
    ax.set_ylabel("count")
    ax.set_title("Pupil-diameter histogram (expected: 2-8 mm)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Pass/fail thresholds

Practical thresholds for filtering recordings before downstream analysis:

- `validity_percent >= 90` - acceptable signal continuity.
- `sample_rate_stability < 0.05` - sub-5% jitter; loosen for prototype rigs.
- `n_gaps == 0` for fixation-sensitive analyses; `<= 5` is fine for
  duration-aggregated metrics.
- `pupil_validity_percent >= 90` - LHIPA needs continuous pupil.

In [ ]:
thresholds = {
    "validity_percent >= 90":          qm.validity_percent >= 90,
    "sample_rate_stability < 0.05":    qm.sample_rate_stability < 0.05,
    "n_gaps <= 5":                     qm.n_gaps <= 5,
    "pupil_validity >= 90":            (qm.pupil_validity_percent or 0) >= 90,
}
for label, ok in thresholds.items():
    mark = "PASS" if ok else "FAIL"
    print(f"  [{mark}] {label}")
print(f"\nComposite quality score: {qm.quality_score:.1f} / 100")